In [1]:
import os
import re
import pandas as pd
import camelot
from PyPDF2 import PdfReader

pdf_to_read = "Remittance_Cenco.pdf"  # Ajusta la ruta si es necesario


In [2]:
reader = PdfReader(pdf_to_read)
num_pages = len(reader.pages)
print("Número de páginas en el PDF:", num_pages)


Número de páginas en el PDF: 26


In [3]:
tables_stream = camelot.read_pdf(pdf_to_read, pages='all', flavor='stream', strip_text='\n')
print("Tablas detectadas con stream:", len(tables_stream))
# Mostrar primeras filas de la primera tabla (si existe)
if tables_stream:
    display(tables_stream[0].df.head())


/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (301.0, 448.924, 438.37600000000003, 567.2204959568734)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (301.0, 448.924, 438.37600000000003, 567.2112062663186)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (301.0, 448.924, 438.37600000000003, 567.215777188329)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables fou

Tablas detectadas con stream: 37


/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (301.0, 448.924, 438.37600000000003, 568.3906666666668)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


,0
0,CENCOSUD COLOMBIA S.A.
1,900.155.107-1
2,AV 9 Nro 125-30 - TESORERIA
3,PAGINA: 1
4,FECHA : 10/09/2025


In [4]:
# Páginas que no detectó stream
stream_pages = set(int(t.page) for t in tables_stream)
all_pages = set(range(1, num_pages+1))
missing_pages = all_pages - stream_pages
print("Páginas no detectadas por stream:", missing_pages)

# Intentar 'lattice' solo en páginas faltantes
tables_lattice = []
if missing_pages:
    missing_pages_str = ",".join(str(p) for p in missing_pages)
    tables_lattice = camelot.read_pdf(pdf_to_read, pages=missing_pages_str, flavor='lattice', strip_text='\n')
    print("Tablas detectadas con lattice:", len(tables_lattice))


Páginas no detectadas por stream: set()


In [5]:
all_tables = list(tables_stream) + list(tables_lattice)
df_all = pd.concat([t.df for t in all_tables], ignore_index=True)
print("Número total de filas concatenadas:", len(df_all))
display(df_all.head(10))


Número total de filas concatenadas: 1453


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,CENCOSUD COLOMBIA S.A.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,900.155.107-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AV 9 Nro 125-30 - TESORERIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PAGINA: 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,FECHA : 10/09/2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,,,,,,,FECHA : 10/09/2025,,,,,,,
6,VOUCHER,DESCRIPCION,DOCUMENTO,TIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,VALOR PAG,DOC.SOPORTE
7,FS,FACTURA VENTA,VPP2 2019435,ADM. JUMBO - SEDE,,01/09/2025,1.899.121.101,348.471.251,0,0,0,75.549.323,2.247.592.352,0
8,DEV,DEVOLUCION MERCANCI AAAA-,,PLAT - CROSSD,DROGUE,25/08/2025,569.389,108.184,0,0,0,0,677.573,0
9,,,0000418742,AVERI,R,,,,,,,,,


In [6]:
header_row_idx = df_all[df_all.apply(lambda r: r.astype(str).str.contains('VOUCHER').any(), axis=1)].index[0]
print("Fila de encabezado detectada:", header_row_idx)

df_all.columns = df_all.iloc[header_row_idx]
df_all = df_all.drop(index=list(range(header_row_idx + 1))).reset_index(drop=True)
df_all = df_all[~df_all.apply(lambda r: all(r.astype(str) == df_all.columns.astype(str)), axis=1)].reset_index(drop=True)

ref_columns = df_all.columns.tolist()
print("Columnas detectadas:", ref_columns)
display(df_all.head(10))


Fila de encabezado detectada: 6
Columnas detectadas: ['VOUCHER', 'DESCRIPCION', 'DOCUMENTO', 'TIENDA', 'SECCION', 'F. REGISTRO', 'VALOR FAC.', 'IVA FAC.', 'RET. FUENTE', 'RET. IVA', 'RET. ICA', 'OTROS IMP.', 'VALOR PAG', 'DOC.SOPORTE']


6,VOUCHER,DESCRIPCION,DOCUMENTO,TIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,VALOR PAG,DOC.SOPORTE
0,FS,FACTURA VENTA,VPP2 2019435,ADM. JUMBO - SEDE,,01/09/2025,1.899.121.101,348.471.251,0,0,0,75.549.323,2.247.592.352,0
1,DEV,DEVOLUCION MERCANCI AAAA-,,PLAT - CROSSD,DROGUE,25/08/2025,569.389,108.184,0,0,0,0,677.573,0
2,,,0000418742,AVERI,R,,,,,,,,,
3,DEV,DEVOLUCION MERCANCI AAAA-,,PLAT - CROSSD,DROGUE,25/08/2025,718.283,136.474,0,0,0,0,854.757,0
4,,,0000418741,AVERI,R,,,,,,,,,
5,DEV,DEVOLUCION MERCANCI AAAA-,,PLAT - CROSS,RANCHO,05/09/2025,298.343,51.405,0,0,0,0,349.748,0
6,,,0000561384,DOCKIN,,,,,,,,,,
7,DEV,DEVOLUCION MERCANCI AAAA-,,PLAT - CROSS,PERFUME,05/09/2025,621.317,118.050,0,0,0,0,739.367,0
8,,,0000561383,DOCKIN,,,,,,,,,,
9,CH,Costo de Transferen,,ADM. JUMBO - SEDE,,10/09/2025,3.000,0,0,0,0,0,3.000,0


In [7]:
# Valores de VOUCHER que nos interesan
filter_values = ['CH','DAV','DCA','DCC','DCF','DEV','DND','DPC','FPM','FS','LTG','RPL','VOUCHER']

# Filtrar solo filas que tengan VOUCHER en la primera columna
filtered_df = df_all[df_all[df_all.columns[0]].isin(filter_values)].copy()

# Función para ordenar
def sort_key(val):
    if val == "VOUCHER": return "0"
    if val == "FPM":     return "1"
    return "2" + str(val)

filtered_df["sort_order"] = filtered_df[filtered_df.columns[0]].apply(sort_key)

In [9]:
# Ordenar, eliminar duplicados y resetear índice
filtered_df = (filtered_df
               .sort_values(by="sort_order")
               .drop(columns="sort_order")
               .drop_duplicates()
               .reset_index(drop=True))

print("Número de filas después de filtrar y ordenar:", len(filtered_df))
display(filtered_df.head(10))

Número de filas después de filtrar y ordenar: 575


6,VOUCHER,DESCRIPCION,DOCUMENTO,TIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,VALOR PAG,DOC.SOPORTE
0,FPM,FACTURA PROVEEDOR,PMP1261242,PLAT - CROSS,PERFUME,11/07/2025,-8.389.770,-1.594.056,0,0,0,0,-9.983.826,0
1,FPM,FACTURA PROVEEDOR,PMP1260709,PLAT - CROSS,PERFUME,11/07/2025,-221.662.784,-42.115.929,0,0,0,0,-263.778.713,0
2,FPM,FACTURA PROVEEDOR,PMP1263885,PLAT - CROSSD,DROGUE,17/07/2025,-4.796.201,-911.278,0,0,0,0,-5.707.479,0
3,FPM,FACTURA PROVEEDOR,PMP1264011,PLAT - CROSSD,PLATOS,18/07/2025,-8.694.169,-1.573.881,0,0,0,0,-10.268.050,0
4,FPM,FACTURA PROVEEDOR,PMP1262826,PLAT - PLAT CROSS,PLATOS,15/07/2025,-1.342.637,-242.868,0,0,0,0,-1.585.505,0
5,FPM,FACTURA PROVEEDOR,PMP1263306,PLAT - PLAT CROSS,DROGUE,16/07/2025,-3.877.916,-736.804,0,0,0,0,-4.614.720,0
6,FPM,FACTURA PROVEEDOR,PMP1260880,PLAT - CROSS,DROGUE,10/07/2025,-3.711.848,-705.251,0,0,0,0,-4.417.099,0
7,FPM,FACTURA PROVEEDOR,PMP1260881,PLAT - CROSS,DROGUE,10/07/2025,-4.129.828,-784.667,0,0,0,0,-4.914.495,0
8,FPM,FACTURA PROVEEDOR,PMP1261044,PLAT - CROSS,PLATOS,10/07/2025,-5.130.067,-916.786,0,0,0,0,-6.046.853,0
9,FPM,FACTURA PROVEEDOR,PMP1261046,PLAT - CROSS,PLATOS,10/07/2025,-5.119.102,-927.998,0,0,0,0,-6.047.100,0


In [11]:
# Renombrar columnas y seleccionar solo las necesarias
remittance = filtered_df.rename(columns={
    "DESCRIPCION": "Tipo de Documento",
    "DOCUMENTO": "Referencia / Factura",
    "VALOR PAG": "Importe de factura"
})[
    ["VOUCHER", "Tipo de Documento", "Referencia / Factura", "Importe de factura", "DOC.SOPORTE", "SECCION"]
]

display(remittance.head(10))


6,VOUCHER,Tipo de Documento,Referencia / Factura,Importe de factura,DOC.SOPORTE,SECCION
0,FPM,FACTURA PROVEEDOR,PMP1261242,-9.983.826,0,PERFUME
1,FPM,FACTURA PROVEEDOR,PMP1260709,-263.778.713,0,PERFUME
2,FPM,FACTURA PROVEEDOR,PMP1263885,-5.707.479,0,DROGUE
3,FPM,FACTURA PROVEEDOR,PMP1264011,-10.268.050,0,PLATOS
4,FPM,FACTURA PROVEEDOR,PMP1262826,-1.585.505,0,PLATOS
5,FPM,FACTURA PROVEEDOR,PMP1263306,-4.614.720,0,DROGUE
6,FPM,FACTURA PROVEEDOR,PMP1260880,-4.417.099,0,DROGUE
7,FPM,FACTURA PROVEEDOR,PMP1260881,-4.914.495,0,DROGUE
8,FPM,FACTURA PROVEEDOR,PMP1261044,-6.046.853,0,PLATOS
9,FPM,FACTURA PROVEEDOR,PMP1261046,-6.047.100,0,PLATOS


In [10]:
remittance = filtered_df.rename(columns={
    filtered_df.columns[0]: "VOUCHER",
    filtered_df.columns[1]: "Tipo de Documento",
    filtered_df.columns[2]: "Referencia / Factura",
    filtered_df.columns[3]: "Importe de factura",
    filtered_df.columns[4]: "DOC.SOPORTE",
    filtered_df.columns[5]: "SECCION"
})[
    ["VOUCHER","Tipo de Documento","Referencia / Factura",
     "Importe de factura","DOC.SOPORTE","SECCION"]
]

display(remittance.head(10))


6,VOUCHER,Tipo de Documento,Referencia / Factura,Importe de factura,DOC.SOPORTE,DOC.SOPORTE,SECCION
0,FPM,FACTURA PROVEEDOR,PMP1261242,PLAT - CROSS,PERFUME,0,11/07/2025
1,FPM,FACTURA PROVEEDOR,PMP1260709,PLAT - CROSS,PERFUME,0,11/07/2025
2,FPM,FACTURA PROVEEDOR,PMP1263885,PLAT - CROSSD,DROGUE,0,17/07/2025
3,FPM,FACTURA PROVEEDOR,PMP1264011,PLAT - CROSSD,PLATOS,0,18/07/2025
4,FPM,FACTURA PROVEEDOR,PMP1262826,PLAT - PLAT CROSS,PLATOS,0,15/07/2025
5,FPM,FACTURA PROVEEDOR,PMP1263306,PLAT - PLAT CROSS,DROGUE,0,16/07/2025
6,FPM,FACTURA PROVEEDOR,PMP1260880,PLAT - CROSS,DROGUE,0,10/07/2025
7,FPM,FACTURA PROVEEDOR,PMP1260881,PLAT - CROSS,DROGUE,0,10/07/2025
8,FPM,FACTURA PROVEEDOR,PMP1261044,PLAT - CROSS,PLATOS,0,10/07/2025
9,FPM,FACTURA PROVEEDOR,PMP1261046,PLAT - CROSS,PLATOS,0,10/07/2025
